# 03_feature_engineering – Creating and Evaluating New Features

**Goal:** Implement the features suggested in EDA and measure their contribution to model performance. We'll use the same cross‑validation framework as in the baseline notebook to ensure fair comparison.

**Features to explore:**
- **Porch/Deck area**, sum of all porch types.
- **Neighborhood aggregates**, mean `SalePrice` per neighborhood (but careful with leakage!).
- **Quality × Condition interactions**, e.g., `OverallQual * OverallCond`.
- **Binned year groups**, decade of construction/remodel.
- **Ratio features**, `LotArea / TotalSF`, `GarageArea / TotalSF`, etc.
- **Other interactions** suggested by correlation patterns.

We'll add these features incrementally and track the log RMSE after each addition.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
import lightgbm as lgb

# For reproducibility
RANDOM_STATE = 42
N_SPLITS = 10

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data (with outlier removal)

In [3]:
data_path = Path("../data/raw/train.csv")
df = pd.read_csv(data_path)
print(f"Original shape: {df.shape}")

# Remove known outliers
outlier_idx = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)].index
df = df.drop(outlier_idx)
print(f"Shape after outlier removal: {df.shape}")

y = df['SalePrice']
X_raw = df.drop(['Id', 'SalePrice'], axis=1)

# Log transform target (we'll work with log scale)
y_log = np.log1p(y)

Original shape: (1460, 81)
Shape after outlier removal: (1458, 81)


# 3. Baseline Performance (without new features)

We need a baseline to compare against. We'll use the same minimal preprocessing pipeline from 01_reproduce_paper.ipynb (median imputation, one‑hot encoding, scaling) and evaluate a Ridge regression model.

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify column types
numeric_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
categoric_cols = X_raw.select_dtypes(include=['object']).columns.tolist()

# Preprocessing pipeline (same as baseline)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categoric_cols)
])

# Ridge model (same alpha as baseline)
model = Ridge(alpha=1.0, random_state=RANDOM_STATE)

# Cross-validation
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
baseline_scores = []

for train_idx, val_idx in kf.split(X_raw):
    X_train, X_val = X_raw.iloc[train_idx], X_raw.iloc[val_idx]
    y_train, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

    preprocessor.fit(X_train)
    X_train_proc = preprocessor.transform(X_train)
    X_val_proc = preprocessor.transform(X_val)

    model.fit(X_train_proc, y_train)
    preds = model.predict(X_val_proc)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    baseline_scores.append(rmse)

print(f"Baseline Ridge CV log RMSE: {np.mean(baseline_scores):.5f} ± {np.std(baseline_scores):.5f}")

Baseline Ridge CV log RMSE: 0.11646 ± 0.01482


# 4. Enhanced Feature Transformer

We'll create an improved version of FeatureTransformer that includes new feature creation methods. To keep things organised, we can define the transformer directly in the notebook (or later move it to features.py). For now, define it here.

In [6]:
from scipy.stats import skew

class EnhancedFeatureTransformer:
    """
    Extended feature transformer with additional engineered features.
    """
    def __init__(self, add_porch=True, add_neighborhood_agg=True,
                 add_quality_interactions=True, add_year_bins=True,
                 add_ratios=True):
        self.add_porch = add_porch
        self.add_neighborhood_agg = add_neighborhood_agg
        self.add_quality_interactions = add_quality_interactions
        self.add_year_bins = add_year_bins
        self.add_ratios = add_ratios

        self.neigh_median_price = None   # to be learned from training data

    def fit(self, X, y=None):
        """Learn any parameters needed for transformations (e.g., neighborhood medians)."""
        if self.add_neighborhood_agg and y is not None:
            # Compute median SalePrice per neighborhood from training data
            df_temp = X.copy()
            df_temp['SalePrice'] = y   # temporary attach target
            self.neigh_median_price = df_temp.groupby('Neighborhood')['SalePrice'].median()
        return self

    def transform(self, X):
        X = X.copy()

        # --- Basic handling (reuse your existing FeatureTransformer logic) ---
        X = self._handle_missing(X)
        X = self._add_basic_features(X)   # TotalSF, TotalBathrooms, HouseAge, RemodelAge, OverallQual_TotalSF

        # --- New features ---
        if self.add_porch:
            X = self._add_porch_features(X)
        if self.add_neighborhood_agg and self.neigh_median_price is not None:
            X = self._add_neighborhood_agg(X)
        if self.add_quality_interactions:
            X = self._add_quality_interactions(X)
        if self.add_year_bins:
            X = self._add_year_bins(X)
        if self.add_ratios:
            X = self._add_ratio_features(X)

        # Convert categoricals
        for col in X.select_dtypes(include='object').columns:
            X[col] = X[col].astype('category')

        return X

    def _handle_missing(self, df):
        # Copy the missing handling from your existing FeatureTransformer
        # (for brevity, I'll summarise – you can copy the full method from features.py)
        none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                     "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                     "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
        for col in none_cols:
            if col in df.columns:
                df[col] = df[col].fillna("None")
        # LotFrontage imputation (you may want to add neighborhood median logic)
        if "LotFrontage" in df.columns:
            df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())
        # Fill remaining numeric with median
        for col in df.select_dtypes(include=np.number).columns:
            df[col] = df[col].fillna(df[col].median())
        return df

    def _add_basic_features(self, df):
        # Reuse your existing feature creation
        if all(c in df.columns for c in ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]):
            df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
        if all(c in df.columns for c in ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]):
            df["TotalBathrooms"] = (df["FullBath"] + 0.5*df["HalfBath"] +
                                     df["BsmtFullBath"] + 0.5*df["BsmtHalfBath"])
        if all(c in df.columns for c in ["YrSold", "YearBuilt"]):
            df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
        if all(c in df.columns for c in ["YrSold", "YearRemodAdd"]):
            df["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]
        if all(c in df.columns for c in ["OverallQual", "TotalSF"]):
            df["OverallQual_TotalSF"] = df["OverallQual"] * df["TotalSF"]
        return df

    def _add_porch_features(self, df):
        porch_cols = ['OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']
        if all(c in df for c in porch_cols):
            df['TotalPorchSF'] = df[porch_cols].sum(axis=1)
        return df

    def _add_neighborhood_agg(self, df):
        # Add median price of neighborhood (learned from training)
        # To avoid leakage, we use the median computed from training data only.
        df['Neighborhood_median_price'] = df['Neighborhood'].map(self.neigh_median_price)
        # Also maybe fill missing test neighborhoods with global median?
        df['Neighborhood_median_price'] = df['Neighborhood_median_price'].fillna(self.neigh_median_price.median())
        return df

    def _add_quality_interactions(self, df):
        if 'OverallQual' in df and 'OverallCond' in df:
            df['Qual_Cond'] = df['OverallQual'] * df['OverallCond']
        if 'OverallQual' in df and 'GrLivArea' in df:
            df['Qual_GrLiv'] = df['OverallQual'] * df['GrLivArea']
        return df

    def _add_year_bins(self, df):
        if 'YearBuilt' in df:
            df['DecadeBuilt'] = (df['YearBuilt'] // 10) * 10
        if 'YearRemodAdd' in df:
            df['DecadeRemod'] = (df['YearRemodAdd'] // 10) * 10
        return df

    def _add_ratio_features(self, df):
        if 'LotArea' in df and 'TotalSF' in df:
            df['LotArea_per_TotalSF'] = df['LotArea'] / (df['TotalSF'] + 1)  # avoid div by zero
        if 'GarageArea' in df and 'TotalSF' in df:
            df['GarageArea_per_TotalSF'] = df['GarageArea'] / (df['TotalSF'] + 1)
        return df

# 5. Evaluate Impact of New Features

We'll use Ridge regression again (or you could use LGBM) to measure performance with the enhanced transformer. We'll create a loop that adds feature groups one by one and records the CV score.

In [7]:
# Define feature groups to test
feature_groups = {
    'baseline': {'add_porch': False, 'add_neighborhood_agg': False,
                 'add_quality_interactions': False, 'add_year_bins': False,
                 'add_ratios': False},
    '+ porch': {'add_porch': True},
    '+ neighborhood_agg': {'add_neighborhood_agg': True},
    '+ quality_interactions': {'add_quality_interactions': True},
    '+ year_bins': {'add_year_bins': True},
    '+ ratios': {'add_ratios': True}
}

# We'll accumulate scores
results = {}

for name, params in feature_groups.items():
    print(f"\n=== Testing: {name} ===")

    # Start with baseline params and update
    current_params = {'add_porch': False, 'add_neighborhood_agg': False,
                      'add_quality_interactions': False, 'add_year_bins': False,
                      'add_ratios': False}
    current_params.update(params)

    transformer = EnhancedFeatureTransformer(**current_params)
    scores = []

    for train_idx, val_idx in kf.split(X_raw):
        X_train, X_val = X_raw.iloc[train_idx], X_raw.iloc[val_idx]
        y_train, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]

        # Fit transformer on training only (pass y_train for neighborhood agg)
        transformer.fit(X_train, y_train)
        X_train_t = transformer.transform(X_train)
        X_val_t = transformer.transform(X_val)

        # Preprocessing for numeric/categorical (now the transformer already did encoding? Actually we need to apply one-hot again? Wait, our transformer returns categoricals as 'category' but we still need to encode for linear models.
        # For simplicity, we'll use the same preprocessor as before, but on the transformed features.
        # But note: transformer may have created new numeric features – we need to re-identify columns.
        # A cleaner approach: let the transformer output a dataframe with numeric and categorical columns, then use the preprocessor again.
        # For now, to keep it simple, we'll just use Ridge on the transformed data after one-hot encoding.
        # We'll reuse the preprocessor defined earlier, but we need to refit it on X_train_t.

        # Identify new column types after transformation
        new_num_cols = X_train_t.select_dtypes(include=[np.number]).columns.tolist()
        new_cat_cols = X_train_t.select_dtypes(include=['category', 'object']).columns.tolist()

        # Create a new preprocessor
        num_trans = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
        cat_trans = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='None')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
        pre = ColumnTransformer([('num', num_trans, new_num_cols),
                                 ('cat', cat_trans, new_cat_cols)])

        pre.fit(X_train_t)
        X_train_proc = pre.transform(X_train_t)
        X_val_proc = pre.transform(X_val_t)

        model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
        model.fit(X_train_proc, y_train)
        preds = model.predict(X_val_proc)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        scores.append(rmse)

    results[name] = {'mean': np.mean(scores), 'std': np.std(scores)}
    print(f"{name} CV log RMSE: {results[name]['mean']:.5f} ± {results[name]['std']:.5f}")


=== Testing: baseline ===
baseline CV log RMSE: 0.11650 ± 0.01519

=== Testing: + porch ===
+ porch CV log RMSE: 0.11650 ± 0.01519

=== Testing: + neighborhood_agg ===
+ neighborhood_agg CV log RMSE: 0.11651 ± 0.01513

=== Testing: + quality_interactions ===
+ quality_interactions CV log RMSE: 0.11660 ± 0.01473

=== Testing: + year_bins ===
+ year_bins CV log RMSE: 0.11658 ± 0.01519

=== Testing: + ratios ===
+ ratios CV log RMSE: 0.11674 ± 0.01503


# 6. Compare Results

In [8]:
result_df = pd.DataFrame(results).T
result_df['improvement'] = result_df['mean'] - result_df.loc['baseline', 'mean']
print(result_df.round(5))

                           mean      std  improvement
baseline                0.11650  0.01519      0.00000
+ porch                 0.11650  0.01519     -0.00000
+ neighborhood_agg      0.11651  0.01513      0.00002
+ quality_interactions  0.11660  0.01473      0.00011
+ year_bins             0.11658  0.01519      0.00008
+ ratios                0.11674  0.01503      0.00024
